# Entrenamiento final XGBoost por jets

Entrena los mejores modelos por `PRI_jet_num`, calibra la politica final de thresholds, compara contra el mejor arbol anterior y genera el bundle/submission final.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_plan_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct = candidate / "xgboost_by_jets_plan_utils.py"
        nested = candidate / "12.plan" / "xgboost_by_jets_plan_utils.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "12.plan"
    raise FileNotFoundError("No se pudo ubicar 12.plan/xgboost_by_jets_plan_utils.py")


plan_dir = find_plan_dir()
if str(plan_dir) not in sys.path:
    sys.path.insert(0, str(plan_dir))

from xgboost_by_jets_plan_utils import (
    PROJECT_ROOT,
    RESULTS_DIR,
    build_validation_predictions,
    dump_json,
    ensure_plan_dirs,
    get_artifacts,
    load_best_params_by_jet,
    reference_best_tree_row,
    save_final_artifacts,
    selected_thresholds_from_policy,
    threshold_policy_comparison,
    train_models_from_params,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
ensure_plan_dirs()
print(f"plan_dir={plan_dir}")

plan_dir=/home/santig14/fing/taa/1_TallerAA/12.plan


## Entrenamiento final

In [2]:
best_params_path = RESULTS_DIR / "06_tuning_xgboost_por_jets_best_params.json"
params_by_jet = load_best_params_by_jet(best_params_path)

artifacts = get_artifacts()
models_by_jet, group_metrics_df = train_models_from_params(artifacts, params_by_jet)
valid_predictions_df = build_validation_predictions(artifacts, models_by_jet)
policy_comparison_df, threshold_details = threshold_policy_comparison(valid_predictions_df)

print("Metricas por jet con los hiperparametros finales:")
display(group_metrics_df.round(6))

print("Politicas de decision final:")
display(policy_comparison_df.round(6))

Metricas por jet con los hiperparametros finales:


,jet_num,train_rows,valid_rows,feature_count,accuracy_default,precision_default,recall_default,f1_default,auc,ams,ams_threshold,fit_seconds
0,0,89987,9926,78,0.826617,0.627406,0.768585,0.690857,0.893698,0.773110,0.880455,4.751252
1,1,69731,7813,74,0.822987,0.771977,0.718214,0.744126,0.890174,0.563862,0.849449,3.822664
2,2,45349,5030,67,0.855070,0.863975,0.852282,0.858088,0.931770,0.650208,0.918633,1.951076
3,3,19933,2231,67,0.847154,0.744928,0.756996,0.750913,0.903803,0.255729,0.729651,1.943627


Politicas de decision final:


,policy,accuracy,precision,recall,f1,auc,ams,signal_weight,background_weight,predicted_signal_count,thresholds
0,joint_thresholds_from_independent,0.78320,0.928863,0.397806,0.557045,0.903617,1.147119,20.956283,316.829615,3669,"{""0"": 0.922630786895752, ""1"": 0.84944850206375..."
1,joint_thresholds_from_global,0.78320,0.928863,0.397806,0.557045,0.903617,1.147119,20.956283,316.829615,3669,"{""0"": 0.922630786895752, ""1"": 0.84944850206375..."
2,independent_threshold_by_jet,0.78648,0.919242,0.413213,0.570140,0.903617,1.118694,25.281364,492.356303,3851,"{""0"": 0.8804548978805542, ""1"": 0.8494485020637..."
3,global_single_threshold,0.78372,0.929115,0.399323,0.558576,0.903617,1.089182,23.865494,462.219868,3682,0.880551397800


## Comparacion contra referencias

In [3]:
best_policy_row = policy_comparison_df.iloc[0]
selected_thresholds = selected_thresholds_from_policy(best_policy_row)
selected_threshold_details = {
    "selected_policy": str(best_policy_row["policy"]),
    "selected_thresholds": selected_thresholds,
    "policy_comparison": policy_comparison_df.to_dict(orient="records"),
    "raw_threshold_details": threshold_details,
}

reference_tree = reference_best_tree_row()
previous_mix_path = PROJECT_ROOT / "11.Mezcla" / "results" / "xgboost_i3_tree_safe_mix_by_jets_vs_best_10try06.csv"
previous_mix_df = pd.read_csv(previous_mix_path)
previous_mix = previous_mix_df.loc[previous_mix_df["dataset"] == "xgboost_i3_tree_safe_mix_by_jets"].iloc[0]

final_row = {
    "dataset": "xgboost_i3_tree_safe_mix_by_jets_final",
    "source": "12.plan final tuning + threshold policy",
    "policy": str(best_policy_row["policy"]),
    "accuracy": float(best_policy_row["accuracy"]),
    "precision": float(best_policy_row["precision"]),
    "recall": float(best_policy_row["recall"]),
    "f1": float(best_policy_row["f1"]),
    "auc": float(best_policy_row["auc"]),
    "ams": float(best_policy_row["ams"]),
    "thresholds": str(best_policy_row["thresholds"]),
}

comparison_df = pd.DataFrame([
    {
        "dataset": str(reference_tree["dataset"]),
        "source": str(reference_tree["source"]),
        "policy": "previous_best_tree_policy",
        "accuracy": float(reference_tree["accuracy"]),
        "precision": float(reference_tree["precision"]),
        "recall": float(reference_tree["recall"]),
        "f1": float(reference_tree["f1"]),
        "auc": float(reference_tree["auc"]),
        "ams": float(reference_tree["ams"]),
        "thresholds": str(reference_tree["ams_threshold"]),
    },
    {
        "dataset": str(previous_mix["dataset"]),
        "source": str(previous_mix["source"]),
        "policy": "11.Mezcla/04 global threshold",
        "accuracy": float(previous_mix["accuracy"]),
        "precision": float(previous_mix["precision"]),
        "recall": float(previous_mix["recall"]),
        "f1": float(previous_mix["f1"]),
        "auc": float(previous_mix["auc"]),
        "ams": float(previous_mix["ams"]),
        "thresholds": str(previous_mix["ams_threshold"]),
    },
    final_row,
])

base_row = comparison_df.iloc[0]
for metric in ["accuracy", "precision", "recall", "f1", "auc", "ams"]:
    comparison_df[f"delta_{metric}_vs_previous_best"] = comparison_df[metric] - float(base_row[metric])

overall_metrics_df = pd.DataFrame([final_row])

print("Comparacion final:")
display(comparison_df.round(6))

Comparacion final:


,dataset,source,policy,accuracy,precision,recall,f1,auc,ams,thresholds,delta_accuracy_vs_previous_best,delta_precision_vs_previous_best,delta_recall_vs_previous_best,delta_f1_vs_previous_best,delta_auc_vs_previous_best,delta_ams_vs_previous_best
0,v2_physics_by_jet,same_best_xgboost_as_7_arboles,previous_best_tree_policy,0.83612,0.743623,0.796311,0.769066,0.909888,1.107879,0.9048761129379272,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
1,xgboost_i3_tree_safe_mix_by_jets,11.Mezcla/03 + XGB por jet,11.Mezcla/04 global threshold,0.79624,0.918535,0.444847,0.599402,0.908090,1.083018,0.8716158866882324,-0.03988,0.174911,-0.351465,-0.169664,-0.001797,-0.024861
2,xgboost_i3_tree_safe_mix_by_jets_final,12.plan final tuning + threshold policy,joint_thresholds_from_independent,0.78320,0.928863,0.397806,0.557045,0.903617,1.147119,"{""0"": 0.922630786895752, ""1"": 0.84944850206375...",-0.05292,0.185240,-0.398506,-0.212021,-0.006271,0.039240


## Export final

In [4]:
submission_label = "xgboost_i3_tree_safe_mix_by_jets_final"

comparison_path = RESULTS_DIR / "08_final_comparison.csv"
group_metrics_path = RESULTS_DIR / "08_final_group_metrics.csv"
policy_path = RESULTS_DIR / "08_final_threshold_policy_comparison.csv"
threshold_details_path = RESULTS_DIR / "08_final_threshold_details.json"
params_path = RESULTS_DIR / "08_final_params_by_jet.json"
summary_path = RESULTS_DIR / "08_final_summary.txt"

comparison_df.to_csv(comparison_path, index=False)
group_metrics_df.to_csv(group_metrics_path, index=False)
policy_comparison_df.to_csv(policy_path, index=False)
dump_json(threshold_details_path, selected_threshold_details)
dump_json(params_path, {"params_by_jet": params_by_jet})

saved_paths = save_final_artifacts(
    artifacts=artifacts,
    models_by_jet=models_by_jet,
    group_metrics_df=group_metrics_df,
    overall_metrics_df=overall_metrics_df,
    threshold_details=selected_threshold_details,
    params_by_jet=params_by_jet,
    submission_label=submission_label,
)

summary_lines = [
    "# Entrenamiento final XGBoost por jets",
    "",
    f"Politica seleccionada: {selected_threshold_details['selected_policy']}",
    f"AMS final validacion: {float(final_row['ams']):.6f}",
    f"AUC final validacion: {float(final_row['auc']):.6f}",
    f"Delta AMS vs mejor previo: {float(comparison_df.iloc[-1]['delta_ams_vs_previous_best']):.6f}",
    f"Delta AUC vs mejor previo: {float(comparison_df.iloc[-1]['delta_auc_vs_previous_best']):.6f}",
    "",
    f"Submission: {saved_paths['submission_path']}",
    f"Bundle: {saved_paths['bundle_dir']}",
]
summary_path.write_text("\n".join(summary_lines) + "\n")

print("Archivos finales:")
for key, value in saved_paths.items():
    print(f"- {key}: {value}")
print(f"- comparison: {comparison_path}")
print(f"- summary: {summary_path}")

Archivos finales:
- bundle_dir: /home/santig14/fing/taa/1_TallerAA/12.plan/models/xgboost_i3_tree_safe_mix_by_jets_final_bundle
- submission_path: /home/santig14/fing/taa/1_TallerAA/12.plan/results/higgs_submission_xgboost_i3_tree_safe_mix_by_jets_final.csv
- test_predictions_path: /home/santig14/fing/taa/1_TallerAA/12.plan/results/xgboost_i3_tree_safe_mix_by_jets_final_test_scores.csv
- validation_path: /home/santig14/fing/taa/1_TallerAA/12.plan/results/xgboost_i3_tree_safe_mix_by_jets_final_submission_validation.json
- comparison: /home/santig14/fing/taa/1_TallerAA/12.plan/results/08_final_comparison.csv
- summary: /home/santig14/fing/taa/1_TallerAA/12.plan/results/08_final_summary.txt
